In [1]:
import pandas as pd
import numpy as np
import random

np.random.seed(42)

# --------------------------
# DIMENSIONS
# --------------------------
dates = pd.date_range(start="2024-01-01", end="2024-12-31", freq="D")

dim_date = pd.DataFrame({
    "date_id": dates,
    "month": dates.strftime("%b"),
    "quarter": "Q" + ((dates.month-1)//3 + 1).astype(str),
    "year": dates.year
})

regions = ["North", "South", "West", "East"]
departments = ["Sales", "Marketing", "Finance", "Operations"]
categories = ["Electronics", "Apparel", "Furniture"]
products = {
    "Electronics": ["Laptop", "Phone", "Tablet"],
    "Apparel": ["Shirt", "Jacket", "Shoes"],
    "Furniture": ["Sofa", "Table", "Chair"]
}
channels = ["Online", "Offline"]
segments = ["Retail", "Wholesale"]

dim_region = pd.DataFrame({
    "region_id": range(1,5),
    "region_name": regions
})

dim_department = pd.DataFrame({
    "department_id": range(1,5),
    "department_name": departments
})

product_list = []
pid = 1
for cat in categories:
    for p in products[cat]:
        product_list.append([pid, p, cat])
        pid += 1

dim_product = pd.DataFrame(product_list, columns=["product_id","product_name","product_category"])

dim_channel = pd.DataFrame({
    "channel_id": range(1,3),
    "channel_name": channels
})

dim_segment = pd.DataFrame({
    "segment_id": range(1,3),
    "segment_name": segments
})

# --------------------------
# BUSINESS LOGIC FUNCTIONS
# --------------------------

def seasonality(month):
    if month in [10,11,12]: return 1.3
    if month in [6,7]: return 0.85
    return 1.0

region_factor = {
    "North": 1.2,
    "South": 1.1,
    "West": 1.0,
    "East": 0.85
}

category_margin = {
    "Electronics": 0.22,
    "Apparel": 0.35,
    "Furniture": 0.28
}

# --------------------------
# GENERATE BASE FACT DATA
# --------------------------

rows = []

for i in range(400):   # more rows = better realism
    date = random.choice(dates)
    month = date.month
    
    region_id = random.randint(1,4)
    product_id = random.randint(1,len(dim_product))
    channel_id = random.randint(1,2)
    segment_id = random.randint(1,2)
    department_id = random.randint(1,4)

    region = dim_region.loc[region_id-1, "region_name"]
    category = dim_product.loc[product_id-1, "product_category"]

    # base revenue
    base = random.randint(20000, 70000)

    # trend factor (growth through year)
    trend = 1 + (date.timetuple().tm_yday / 365) * 0.2

    revenue = base * trend * seasonality(month) * region_factor[region]
    revenue = revenue * np.random.normal(1, 0.08)  # noise

    revenue = max(5000, revenue)

    # margin logic
    margin = category_margin[category] + np.random.normal(0, 0.05)
    margin = np.clip(margin, 0.05, 0.5)

    gross_profit = revenue * margin
    cogs = revenue - gross_profit

    operating_expenses = revenue * random.uniform(0.1, 0.3)

    net_profit = gross_profit - operating_expenses

    # allow some losses (important realism)
    if random.random() < 0.1:
        net_profit *= -0.5

    sales_volume = int(revenue / random.uniform(200, 800))

    rows.append([
        date, region_id, product_id, channel_id, segment_id,
        department_id, sales_volume, revenue, cogs,
        operating_expenses, net_profit
    ])

base_df = pd.DataFrame(rows, columns=[
    "date_id","region_id","product_id","channel_id","segment_id",
    "department_id","sales_volume","revenue","cogs",
    "operating_expenses","net_profit"
])

# --------------------------
# SPLIT FACT TABLES
# --------------------------

fact_revenue = base_df[[
    "date_id","region_id","product_id","channel_id","segment_id",
    "sales_volume","revenue"
]]

fact_expenses = base_df[[
    "date_id","department_id","cogs","operating_expenses"
]]

# Budget table (annual planning)
fact_budget = dim_department.copy()
fact_budget["budgeted_revenue"] = [500000, 450000, 350000, 400000]
fact_budget["budgeted_expenses"] = [300000, 280000, 220000, 250000]

# --------------------------
# SAVE FILES
# --------------------------

dim_date.to_csv("D:/Documents/MBA/IIP Skills updrade courses/Projects/Power BI/Financial KPI Monitoring & Profitability Dashboard/dim_date.csv", index=False)
dim_region.to_csv("D:/Documents/MBA/IIP Skills updrade courses/Projects/Power BI/Financial KPI Monitoring & Profitability Dashboard/dim_region.csv", index=False)
dim_department.to_csv("D:/Documents/MBA/IIP Skills updrade courses/Projects/Power BI/Financial KPI Monitoring & Profitability Dashboard/dim_department.csv", index=False)
dim_product.to_csv("D:/Documents/MBA/IIP Skills updrade courses/Projects/Power BI/Financial KPI Monitoring & Profitability Dashboard/dim_product.csv", index=False)
dim_channel.to_csv("D:/Documents/MBA/IIP Skills updrade courses/Projects/Power BI/Financial KPI Monitoring & Profitability Dashboard/dim_channel.csv", index=False)
dim_segment.to_csv("D:/Documents/MBA/IIP Skills updrade courses/Projects/Power BI/Financial KPI Monitoring & Profitability Dashboard/dim_segment.csv", index=False)

fact_revenue.to_csv("D:/Documents/MBA/IIP Skills updrade courses/Projects/Power BI/Financial KPI Monitoring & Profitability Dashboard/fact_revenue.csv", index=False)
fact_expenses.to_csv("D:/Documents/MBA/IIP Skills updrade courses/Projects/Power BI/Financial KPI Monitoring & Profitability Dashboard/fact_expenses.csv", index=False)
fact_budget.to_csv("D:/Documents/MBA/IIP Skills updrade courses/Projects/Power BI/Financial KPI Monitoring & Profitability Dashboard/fact_budget.csv", index=False)

print("All SQL-ready datasets generated successfully.")

All SQL-ready datasets generated successfully.
